In [0]:
%sql
DESCRIBE electricity.bronze.production

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window


frames = []
for ds in ["production", "consumption", "wind"]:
    frames.append(
        spark.table(f"electricity.bronze.{ds}").select(
            F.col("datasetId").cast("int").alias("dataset_id"),
            F.lit(ds).alias("measure"),
            F.col("startTime").cast("timestamp").alias("start_time_utc"),
            F.col("endTime").cast("timestamp").alias("end_time_utc"),
            F.col("value").cast("double").alias("value_mw"),
            "source_file",
            "ingested_at",
        )
    )

silver = frames[0].unionByName(frames[1]).unionByName(frames[2])

silver = (silver
    .withColumn("had_stray_seconds", F.second("start_time_utc") != 0)
    .withColumn("start_time_utc", F.date_trunc("minute", "start_time_utc"))
    .withColumn("end_time_utc",   F.date_trunc("minute", "end_time_utc"))
    .withColumn("resolution_min",
        ((F.unix_timestamp("end_time_utc") - F.unix_timestamp("start_time_utc")) / 60).cast("int"))
    .withColumn("start_time_fi",
        F.from_utc_timestamp("start_time_utc", "Europe/Helsinki"))
    .filter(F.col("start_time_utc").isNotNull() & F.col("value_mw").isNotNull())
)

w = Window.partitionBy("dataset_id", "start_time_utc").orderBy("had_stray_seconds")
silver = (silver
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn", "had_stray_seconds")
)

silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("electricity.silver.grid_observations")

In [0]:
%sql
SELECT measure, resolution_min, COUNT(*) AS row_count,
       MIN(start_time_utc) AS first_time, MAX(start_time_utc) AS last_time
FROM electricity.silver.grid_observations
GROUP BY measure, resolution_min
ORDER BY measure, resolution_min;

In [0]:
%sql
select count(*) from electricity.silver.grid_observations

In [0]:
%sql
WITH ordered AS (
  SELECT measure, start_time_utc, end_time_utc,
         LEAD(start_time_utc) OVER (PARTITION BY measure ORDER BY start_time_utc) AS next_start
  FROM electricity.silver.grid_observations
)
SELECT measure,
       end_time_utc AS gap_from,
       next_start   AS gap_to,
       timestampdiff(MINUTE, end_time_utc, next_start) AS gap_minutes
FROM ordered
WHERE next_start <> end_time_utc
ORDER BY measure, gap_from;

In [0]:
%sql
WITH ordered AS (
  SELECT measure, start_time_utc, end_time_utc,
         LEAD(start_time_utc) OVER (PARTITION BY measure ORDER BY start_time_utc) AS next_start
  FROM electricity.silver.grid_observations
)
SELECT measure, start_time_utc, end_time_utc, next_start
FROM ordered
WHERE next_start <> end_time_utc
  AND timestampdiff(MINUTE, end_time_utc, next_start) IN (0, -14)
ORDER BY start_time_utc
LIMIT 20;